In [1]:
# run_boot.py - chạy sau khi đã lưu Khung 4.8 thành boot_metrics.py (cùng thư mục)
import time
from itertools import combinations
import numpy as np
import pandas as pd
from boot_metrics import paired_bootstrap, strata

PARQUET_DIR = "./parquet"
N_BOOT = 1000
PAIRS = [("A", "B"), ("A", "C"), ("B", "C")]      # (x, y) -> Δ = y - x
RUN_STRATA = True                                  # đặt False nếu muốn chạy phần 1, 2 trước


def load(mode, fold, seed):
    return pd.read_parquet(f"{PARQUET_DIR}/DKT_{mode}_f{fold}_s{seed}.parquet")


def run_one(fold, seed):
    runs = {k: load(k, fold, seed) for k in "ABC"}
    out = []
    for x, y in PAIRS:
        t = time.time()
        r = paired_bootstrap(runs[x], runs[y], n_boot=N_BOOT, seed=0)
        r.insert(0, "so sánh", f"{y} − {x}")
        r.insert(0, "seed", seed)
        r.insert(0, "fold", fold)
        out.append(r)
        print(f"fold {fold} seed {seed}: {y} − {x} xong sau {time.time()-t:.0f}s", flush=True)
    return pd.concat(out, ignore_index=True), runs


# ---------- Phần 1: seed 42, cả 5 fold ----------
res = []
runs_s42 = {}
for fold in range(5):
    r, runs = run_one(fold, 42)
    res.append(r)
    runs_s42[fold] = runs
df42 = pd.concat(res, ignore_index=True)
df42["khác biệt có ý nghĩa"] = (df42["lo"] > 0) | (df42["hi"] < 0)
df42.to_csv("boot_all_s42.csv", index=False)
print("\n=== Seed 42, 5 fold ===")
print(df42.round(4).to_string(index=False))
# ΔAUC ở đây phải trùng bảng cũ (cùng seed bootstrap = 0)

# ---------- Phần 2: fold 0, seed 42/43/44 ----------
res = [df42[df42["fold"] == 0]]
for seed in (43, 44):
    r, _ = run_one(0, seed)
    res.append(r)
dfs = pd.concat(res, ignore_index=True)
dfs.to_csv("boot_fold0_seeds.csv", index=False)

print("\n=== Fold 0 qua các seed ===")
summ = (dfs.groupby(["so sánh", "metric"])
        .agg(delta_mean=("delta", "mean"),
             delta_std=("delta", "std"),
             n_cung_dau=("delta", lambda s: int((np.sign(s) == np.sign(s.mean())).sum())),
             n_seed=("delta", "size"))
        .reset_index())
print(summ.round(5).to_string(index=False))
print("\nChi tiết từng seed (C − A):")
print(dfs[dfs["so sánh"] == "C − A"].round(4).to_string(index=False))

# ---------- Phần 3: phân tích theo nhóm câu (C − A, seed 42) ----------
if RUN_STRATA:
    rows = []
    for fold in range(5):
        qa, qc = runs_s42[fold]["A"], runs_s42[fold]["C"]
        assert "n_kc" in qa.columns, "preds thiếu cột n_kc: merge từ dữ liệu gốc theo qidx trước"
        s = strata(qa)
        for col in ("cau", "cau_truoc"):
            for v, part in s.groupby(col):
                r = paired_bootstrap(qa, qc, n_boot=N_BOOT, seed=0, subset=part["qidx"])
                r.insert(0, "nhóm", f"{col}={v}")
                r.insert(0, "fold", fold)
                rows.append(r)
                print(f"fold {fold} {col}={v} xong", flush=True)
    dfg = pd.concat(rows, ignore_index=True)
    dfg.to_csv("boot_strata_s42.csv", index=False)
    print("\n=== Theo nhóm câu, C − A, seed 42 ===")
    print(dfg.round(4).to_string(index=False))

fold 0 seed 42: B − A xong sau 80s
fold 0 seed 42: C − A xong sau 81s
fold 0 seed 42: C − B xong sau 85s
fold 1 seed 42: B − A xong sau 82s
fold 1 seed 42: C − A xong sau 84s
fold 1 seed 42: C − B xong sau 84s
fold 2 seed 42: B − A xong sau 81s
fold 2 seed 42: C − A xong sau 81s
fold 2 seed 42: C − B xong sau 82s
fold 3 seed 42: B − A xong sau 97s
fold 3 seed 42: C − A xong sau 81s
fold 3 seed 42: C − B xong sau 82s
fold 4 seed 42: B − A xong sau 81s
fold 4 seed 42: C − A xong sau 80s
fold 4 seed 42: C − B xong sau 80s

=== Seed 42, 5 fold ===
 fold  seed so sánh metric   delta      lo      hi   n_cau  khác biệt có ý nghĩa
    0    42   B − A   ΔAUC  0.0003  0.0001  0.0005 1099197                  True
    0    42   B − A ΔBrier -0.0002 -0.0002 -0.0001 1099197                  True
    0    42   B − A   ΔECE -0.0016 -0.0022 -0.0010 1099197                  True
    0    42   C − A   ΔAUC  0.0053  0.0048  0.0057 1099197                  True
    0    42   C − A ΔBrier -0.0017 -0.0018 -0